## **CREACION DEL ENTORNO**

1.1 Creamos nuestra esturctura dentro de Unity Catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS sesion1;
CREATE SCHEMA IF NOT EXISTS sesion1.data COMMENT 'This is customer catalog';
CREATE VOLUME IF NOT EXISTS sesion1.data.landing;

1.2 Incorporacion de Datos a un Volumen

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/nutrients_csvfile.csv -o /Volumes/sesion1/data/landing/nutrients_csvfile.csv

1.3 Crear una Tabla

In [0]:
%sql
CREATE OR REPLACE TABLE sesion1.data.tabla_simple (
  id INT,
  letra STRING,
  valor DOUBLE
);

INSERT INTO sesion1.data.tabla_simple VALUES
  (1, 'A', 10.5),
  (2, 'B', 20.0),
  (3, 'C', 30.75);

In [0]:
%sql
Select * from sesion1.data.tabla_simple;

1.4 Crear una Vista

In [0]:
%sql
CREATE OR REPLACE VIEW sesion1.data.vw_tabla_simple AS
SELECT
  id,
  letra
FROM sesion1.data.tabla_simple
WHERE valor > 15;

In [0]:
%sql
select * from sesion1.data.vw_tabla_simple

1.5 Crear una función

In [0]:
%sql
create schema if not exists sesion1.security;

In [0]:
%sql
CREATE OR REPLACE FUNCTION sesion1.security.fn_mayor_que_10(x DOUBLE)
RETURNS BOOLEAN
RETURN x > 12;

In [0]:
%sql
SELECT *
FROM sesion1.data.tabla_simple
WHERE sesion1.security.fn_mayor_que_10(valor);

1.6 Creacion de DataFrame

In [0]:
path_data_demo = "/Volumes/sesion1/data/landing/data_demo.csv"
df = spark.read.csv(
    path=path_data_demo,
    header=True,     
    inferSchema=True
)

1.7 Visualizar el Data Frame

In [0]:
display(df)

### 2. Delta Tables

2.1 Subir un CSV al volumen

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/ventas_2025.csv -o /Volumes/sesion1/data/landing/ventas.csv


2.2 Crear el Data Frame

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

base_path = "dbfs:/Volumes/sesion1/data/landing/"

# Definir el esquema manualmente
schema = StructType([
    # Nombre, Tipo de dato, Requerido
    StructField("id", StringType(), True),
    StructField("fecha", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio", DoubleType(), True)
])

df_csv = spark.read.csv(
    path=base_path+"ventas.csv",
    header=True,
    schema=schema,
    sep=","
)

2.3 Creamos la tabla Delta

In [0]:

df_csv.write.format("delta").mode("overwrite").saveAsTable("sesion1.data.productos")

2.4 Leer Tabla Delta

In [0]:
%sql
SELECT * FROM sesion1.data.productos;

2.4.1 Lectura de la tabla Delta con DF de Spark

In [0]:
df_delta = spark.read.table("sesion1.data.productos")

df_delta.show()

2.5 Modificar una Tabla

In [0]:
%sql
-- INSERT: añadir un nuevo registro
INSERT INTO sesion1.data.productos (id, fecha, producto, cantidad, precio)
VALUES ('A001', '2025-11-25', 'Torre E-138', 3, 120000.50);

2.5.1 Update

In [0]:
%sql
-- UPDATE (modify): modificar un registro existente
UPDATE sesion1.data.productos
SET cantidad = 5,
    precio   = 119000.00
WHERE id = '1';

2.5.2 Delete

In [0]:
%sql
-- DELETE: eliminar un registro
DELETE FROM sesion1.data.productos
WHERE id = 'A001';

2.5.3 Merge

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "sesion1.data.productos")  

# Nuevos datos a insertar/actualizar
columns = ["id", "fecha", "producto", "cantidad", "precio"]

nuevos_datos = [(3, "2025-05-24", "Monitor", 1, 179.99), (4, "2025-05-24", "Impresora", 2, 89.99)]
df_updates = spark.createDataFrame(nuevos_datos, columns)


delta_table.alias("target").merge(
    df_updates.alias("source"),
    "target.id = source.id") \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

2.6 Time Travel

2.6.1 Ver el historial de versiones de la tabla

In [0]:
%sql
DESCRIBE HISTORY sesion1.data.productos;

2.6.2 consultar versión anterior (Time Travel por versión)

In [0]:
%sql
SELECT * FROM sesion1.data.productos VERSION AS OF 15;

2.6.3 Consultar por TimeStamp

In [0]:
%sql
SELECT * FROM sesion1.data.productos VERSION AS OF 19;

2.6.4 Restaurar la versión anterior

In [0]:
%sql
CREATE OR REPLACE TABLE sesion1.data.productos_v3
AS SELECT * FROM sesion1.data.productos VERSION AS OF 19;

3 Introducción a PySpark

3.0 Preparación entorno

In [0]:
%sql
create catalog if not exists sesion1;
create schema if not exists sesion1.sparkintro;
create volume if not exists sesion1.sparkintro.landing;

3.1 Almacenar datos en el volumen

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/spark_intro.csv -o /Volumes/sesion1/sparkintro/landing/spark_intro.csv

%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/dim_spark_intro.csv -o /Volumes/sesion1/sparkintro/landing/dim_spark_intro.csv

3.2 Crear el DF

In [0]:
from pyspark.sql import functions as F

# Read CSV from volume as a Spark DataFrame
sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)  # Let Spark infer column types for now
    .csv("/Volumes/sesion1/sparkintro/landing/spark_intro.csv")
)

display(sales_df)

3.3 Select(): elegir columnas

In [0]:
# Select a subset of columns
sales_simple_df = sales_df.select("order_id", "order_date", "country", "units_sold")
display(sales_simple_df)

3.4 filter() / where(): filtrar filas

In [0]:
# Filter sales only for Spain
spain_sales_df = sales_df.filter(F.col("country") == "Spain")
display(spain_sales_df)

# Filter sales with more than 3 units sold
big_orders_df = sales_df.filter(F.col("units_sold") > 3)
display(big_orders_df)

# The previous error occurred because this line is misplaced:
# '3.4 filter() / where(): filtrar filas' should NOT be part of the code cell, so it is removed.


3.5 withColumn(): crear o transformar columnas

In [0]:
from pyspark.sql import functions as F

# Create a new column with total sales amount
sales_with_total_df = sales_df.withColumn(
    "total_sales",
    F.col("units_sold") * F.col("unit_price")  # Simple numeric expression
)

display(sales_with_total_df)

3.6 groupBy().agg(): agregaciones

In [0]:
# Aggregate total units and total sales by country
sales_country_df = (
    sales_with_total_df
    .groupBy("country")
    .agg(
        F.sum("units_sold").alias("total_units"),
        F.sum("total_sales").alias("total_sales_amount")
    )
)

display(sales_country_df)

3.7 orderBy(): ordenar datos

In [0]:
# Order countries by total sales amount (descending)
sorted_sales_country_df = sales_country_df.orderBy(F.col("total_sales_amount").desc())
display(sorted_sales_country_df)

3.8 join(): combinar datos de distintas tablas

In [0]:
# Read product dimension from volume
product_dim_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/sesion1/sparkintro/landing/dim_spark_intro.csv")
)

display(product_dim_df)

In [0]:
# Join sales with product dimension on 'product'
sales_enriched_df = (
    sales_with_total_df.alias("s")
    .join(
        product_dim_df.alias("p"),
        on="product",   # Join key
        how="left"     # Keep all sales even if some product has no dimension row
    )
)

display(sales_enriched_df)

3.9 Write(): Guardar el resultado

In [0]:
# Save aggregated sales by country as a managed Delta table
(
    sorted_sales_country_df
    .write
    .mode("overwrite")          # Overwrite existing table if it exists
    .saveAsTable("sesion1.sparkintro.sales_by_country")
)

# Check that we can read it back as a table
result_df = spark.table("sesion1.sparkintro.sales_by_country")
display(result_df)


### 4. Analisis Exploratorio con Spark

4.1 Dimension, Tipos de Datos y primeras filas

In [0]:
# Número de filas y columnas
print(f"Dimensiones del DataFrame: {df.count()} filas, {len(df.columns)} columnas")

# Tipos de datos
df.printSchema()

# Mostrar las primeras filas
df.show(5)

4.2 Verificacion de Nulos

In [0]:
from pyspark.sql.functions import isnan, when, count, col

display(
    df.select([
    count(
        when(
            isnan(c), c
        )
    ).alias(c) for c in df.columns])
)

**4.3 Info Estadistica**

4.3.1 Metodo Describe

In [0]:

df.describe().show()

4.3.2 Metodo Summary

In [0]:
display(df.summary())

4.3.3 Matriz de Correlación

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Seleccionar columnas numéricas de Spark
numeric_cols = [c for c, t in df.dtypes if t in ("int", "bigint", "double", "float", "decimal")]

# 2. Crear columna vectorial para Spark ML
assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="features"
)
df_vector = assembler.transform(df).select("features")

# 3. Calcular la matriz de correlación (Pearson) con Spark
corr_matrix_row = Correlation.corr(df_vector, "features", "pearson").head()
corr_matrix = corr_matrix_row[0].toArray()  # DenseMatrix -> NumPy array

# 4. Pasar la matriz a Pandas para visualizar
corr_df = pd.DataFrame(corr_matrix, index=numeric_cols, columns=numeric_cols)

# 5. Dibujar el heatmap (igual que antes)
pd.set_option('display.max_columns', None)

plt.figure(figsize=(10, 8))
mask_heatmap = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(
    data=corr_df,
    annot=True,
    linewidth=3,
    cmap='Blues',
    mask=mask_heatmap
)
plt.show()

4.3.4 Conteo de Outliers sobre la columna "time on market"

In [0]:
from pyspark.sql import functions as F

# Calcular Q1 y Q3 usando approxQuantile (eficiente en Spark distribuido)
q1, q3 = df.approxQuantile("time_on_market", [0.25, 0.75], 0.05)

# Calcular IQR
iqr = q3 - q1

# Definir límites inferior y superior usando la regla clásica
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Filtrar los outliers fuera de los límites
outliers = df.filter(
    (F.col("time_on_market") < lower_bound) | 
    (F.col("time_on_market") > upper_bound)
)

# Mostrar los valores calculados
print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

# Mostrar los outliers detectados
display(outliers)

#Conteo de outliers
print(f"Total de outliers detectados: {outliers.count()}")

4.3.5 Z-Score

In [0]:
from pyspark.sql import functions as F

# Calcular la media y la desviación estándar
mean_stddev = df.select(
    F.mean("time_on_market").alias("mean"),
    F.stddev("time_on_market").alias("stddev")
).first()

mean_value = mean_stddev["mean"]
stddev_value = mean_stddev["stddev"]

# Calcular el Z-score
df_with_zscore = df.withColumn(
    "zscore_time_on_market",
    (F.col("time_on_market") - mean_value) / stddev_value
)

# Filtrar los outliers
outliers = df_with_zscore.filter(
    (F.abs(F.col("zscore_time_on_market")) > 3)
)

# Mostrar los outliers
display(outliers)

### 5 DButils

5.1 Obtenermos productos unicos

In [0]:
# Cargar la tabla Delta registrada en el metastore
df = spark.table("sesion1.data.productos")

# Obtener valores únicos de la columna 'producto'
lista_productos = [row['producto'] for row in df.select("producto").distinct().collect()]

lista_productos

5.2 Creamos un widget Dropdown

In [0]:
dbutils.widgets.dropdown(
    name="producto",
    defaultValue=lista_productos[0],   # First element as default
    choices=lista_productos
)

5.3 Creamos un widget text para rango minimo de precios

In [0]:
dbutils.widgets.text("min_price", "0", "Precio mínimo")

5.4 Consulta con paranetros de Widget

5.4.1 Con PySpark

In [0]:
from pyspark.sql.functions import col, avg

# Get widget values
producto_sel = dbutils.widgets.get("producto")          # dropdown
min_price = float(dbutils.widgets.get("min_price"))     # text

# Load the table
df = spark.table("sesion1.data.productos")

# Apply filters
df_filtrado = (
    df
    .filter(col("producto") == producto_sel)
    .filter(col("precio") >= min_price)
)

# Calculate average price
df_media = df_filtrado.agg(avg("precio").alias("precio_medio"))

display(df_media)

5.4.2 Con SQL

In [0]:
%sql
SELECT 
  AVG(precio) AS precio_medio
FROM 
  sesion1.data.productos
WHERE 
  producto = :producto
  AND precio >= CAST(:min_price AS DOUBLE)

### 6 Creacion de Dashboard con SQL

6.1 Visualizaciones rápidas

In [0]:
%sql
SELECT * FROM sesion1.data.productos;

Databricks visualization. Run in Databricks to view.

**6.2 Creacion de un Dashboard**

In [0]:
%sql
-- 1. Crear la tabla con nombres y tipos de columnas
CREATE OR REPLACE TABLE sesion1.data.dim_producto (
    product_id       INT,
    nombre_producto  STRING,
    categoria        STRING,
    tipo_dispositivo STRING,
    es_periferico    BOOLEAN,
    es_entrada       BOOLEAN,
    es_salida        BOOLEAN
)
USING DELTA;

-- 2. Insertar los datos
INSERT INTO sesion1.data.dim_producto VALUES
    (1,  'Mouse',        'Accesorios',     'Entrada',         TRUE,  TRUE,  FALSE),
    (2,  'Teclado',      'Accesorios',     'Entrada',         TRUE,  TRUE,  FALSE),
    (3,  'Webcam',       'Multimedia',     'Entrada/Salida',  TRUE,  TRUE,  TRUE),
    (4,  'Microfono',    'Multimedia',     'Entrada',         TRUE,  TRUE,  FALSE),
    (5,  'Altavoces',    'Multimedia',     'Salida',          TRUE,  FALSE, TRUE),
    (6,  'Disco Duro',   'Almacenamiento', 'Almacenamiento',  FALSE, FALSE, FALSE),
    (7,  'Auriculares',  'Multimedia',     'Entrada/Salida',  TRUE,  TRUE,  TRUE),
    (8,  'USB',          'Accesorios',     'Almacenamiento',  FALSE, FALSE, FALSE),
    (9,  'Impresora',    'Oficina',        'Salida',          TRUE,  FALSE, TRUE),
    (10, 'Monitor',      'Oficina',        'Salida',          TRUE,  FALSE, TRUE);

6.2.1 Creacion de un Dashboard mediante consulta